# Task B Run 23 -- Gemma-4-12B: four epochs, more seeds, final submission

Run 21 put Gemma-4-12B (QLoRA, 3 epochs, two seeds) at **0.6792** on the 530-row holdout
against MuRIL's 0.6151. Its logs point at two cheap gains:

- **Epochs.** Both seeds were still rising at the last epoch: 0.597, 0.629, 0.657 and
  0.561, 0.630, 0.661.
- **Seeds.** Averaging two seeds gave about +2 points over either seed alone: 0.657 and
  0.661 became 0.679.

**Stage 1, epochs, on the same holdout.** Seeds 42 and 43 are trained for **4 epochs**, one
per T4, about 1.6 h. They are compared with Run 21's 3-epoch seeds 42 and 43, loaded from
its output, as three candidates:

| candidate | models averaged |
|---|---|
| `3ep` | Run 21's two 3-epoch models |
| `4ep` | the two new 4-epoch models |
| `mix` | all four |

Each is scored by macro-F1, with paired bootstraps against `3ep`. The highest holdout
macro-F1 is chosen. The choice is made on this holdout, so the chosen score is slightly
optimistic; the bootstrap lines show whether the gap is more than noise.

**Stage 2, full fits on all 3,532 rows.** Every option ends with four models averaged:

| choice | new full fits | reused from Run 21 | models averaged |
|---|---|---|---|
| `3ep` | 3-epoch seeds 44, 45 | 3-epoch seeds 42, 43 | 4 |
| `4ep` | 4-epoch seeds 42, 43, 44 | -- | 3, plus a 4th seed if the 11 h budget allows |
| `mix` | 4-epoch seeds 42, 43 | 3-epoch seeds 42, 43 | 4 |

**Stage 3.** Writes `RECOMMENDED_b23_llm.zip` for the released test file, and reports
agreement with Run 21's LLM submission.

**Before running:**
- **Add Input -> Notebook Output -> your Run 21 notebook.** This is required: Run 21's
  3-epoch holdout and full-fit probabilities are reused.
- **Accelerator** `GPU T4 x2`, **Internet** on. No token is needed.
- **Save & Run All.** The run takes about 5.5-7 h. A hard stop at 11 h means the notebook
  always finishes and packages whatever full fits completed.

In [ ]:
import glob, json, os, pathlib, shutil, subprocess, sys

REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"
WORK = "/kaggle/working/hastika"


def git(*args):
    return subprocess.run(["git", "-C", WORK, *args], check=True,
                          capture_output=True, text=True).stdout.strip()


# Never trust an existing clone (the Run 18/21 failures): resolve the remote tip first,
# so "Internet off" fails here with a clear message, then hard-reset to it.
try:
    remote = subprocess.run(["git", "ls-remote", "--exit-code", REPO, f"refs/heads/{BRANCH}"],
                            check=True, capture_output=True, text=True,
                            timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit("Cannot reach GitHub. In Kaggle turn on Settings -> Internet (needs a "
                     "phone-verified account), then restart the session.") from e
if os.path.isdir(WORK + "/.git"):
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)
head = git("rev-parse", "HEAD")
assert head == remote, f"clone is at {head[:8]} but {BRANCH} is at {remote[:8]}"
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
print("repo at", git("log", "-1", "--oneline"))

# system environment: the repo's MuRIL trainer, exactly as Run 19 installed it
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret loaded")
except Exception:
    print("no HF_TOKEN secret: gated models (Gemma) will be skipped, the rest still run")

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "select GPU T4 x2"
N_GPU = torch.cuda.device_count()
print("gpus:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
if N_GPU < 2:
    print("WARNING: one GPU only -- jobs run one after another and take about twice as long")
print(f"free disk on /tmp: {shutil.disk_usage('/tmp').free / 2**30:.0f} GB")

OUT = pathlib.Path("/kaggle/working/b23_outputs")
RUNS = OUT / "runs"
LOGS = OUT / "logs"
for p in (RUNS, LOGS):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = "/tmp/hf"          # model downloads stay off the 20 GB /kaggle/working limit
os.environ["HF_HUB_CACHE"] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import (LABELS, TEST_FILE, fingerprint, paired_bootstrap,
                                             report, split)

# The split is committed to the repo, so it cannot drift with a library version on Kaggle.
SPLIT = pathlib.Path(WORK) / "data/derived/task_b_combined_holdout"
df = pd.read_csv(SPLIT / "all.csv")
holdout = pd.read_csv(SPLIT / "holdout.csv")
assert len(df) == 3532 and len(holdout) == 530, (len(df), len(holdout))
HOLDOUT_FP = fingerprint(holdout["id"])
assert HOLDOUT_FP == "f85f4f049b", HOLDOUT_FP      # same rows in every notebook
check_df, _, ho_check = split()
if fingerprint(check_df["id"].iloc[ho_check]) != HOLDOUT_FP:
    print("note: recomputing the split here gives different rows (library versions); "
          "the committed split is used")
y_ho = holdout["Hate Category"].map(LABELS.index).to_numpy()
TEST_CSV = pathlib.Path("/kaggle/working/test_inputs.csv")
pd.read_csv(TEST_FILE)[["id", "Comment"]].to_csv(TEST_CSV, index=False)
print(f"train {len(df) - len(holdout)}, holdout {len(holdout)} ({HOLDOUT_FP}), test {len(pd.read_csv(TEST_CSV))}")

In [ ]:
# A separate environment for the LLM stack: Gemma-4 needs transformers >= 5.12, the
# repo's MuRIL trainer is pinned below that. --system-site-packages reuses Kaggle's torch
# and CUDA; --without-pip avoids ensurepip, which Kaggle's Python lacks, and the
# system pip installs into the venv because it runs under the venv's interpreter.
LLM_PY = "/tmp/llmenv/bin/python"
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip",
                    "/tmp/llmenv"], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" '
               f'"bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf',
               shell=True, check=True)
subprocess.run([LLM_PY, "-c", "import torch, transformers, peft, bitsandbytes; "
                "print('LLM env: torch', torch.__version__, '| transformers', "
                "transformers.__version__, '| peft', peft.__version__, '| bnb', "
                "bitsandbytes.__version__)"], check=True)


def slug(model):
    return model.split("/")[-1].lower()


def download(model):
    """Weights only (safetensors), once, before parallel jobs read the same cache."""
    from huggingface_hub import snapshot_download
    snapshot_download(model, cache_dir=HF_CACHE, token=os.environ.get("HF_TOKEN") or None,
                      allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt",
                                      "tokenizer*", "*.jinja"])
    print(f"downloaded {model}; {shutil.disk_usage('/tmp').free / 2**30:.0f} GB free")


def llm_job(model, name, train, predict, seed, evalf="", extra=""):
    ev = f"--eval {evalf}" if evalf else ""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_classifier --model {model} "
                    f"--train {train} {ev} --predict {' '.join(map(str, predict))} "
                    f"--out {RUNS / name} --seed {seed} --cache {HF_CACHE} {extra}")}


def run_with_oom_retry(jobs):
    """One retry at half the micro-batch (same effective batch) for any job that ran
    out of GPU memory. Exit code 3 is fp16 overflow: not retried, reported."""
    codes = run_queue(jobs, N_GPU, raise_on_fail=False)
    retry = []
    for j in jobs:
        if codes.get(j["name"]) and "OutOfMemoryError" in open(j["log"], errors="replace").read():
            print(f"retrying {j['name']} with --bs 4 --grad-accum 4 after OOM")
            retry.append({**j, "cmd": j["cmd"] + " --bs 4 --grad-accum 4 --eval-bs 8"})
    if retry:
        codes.update(run_queue(retry, N_GPU, raise_on_fail=False))
    for n, c in codes.items():
        if c == 3:
            print(f"{n}: fp16 overflow, this model cannot train on a T4")
    return codes

In [ ]:
import time
T_START = time.time()
HARD_STOP = T_START + 11 * 3600
MODEL = "google/gemma-4-12B"
S = slug(MODEL)

found = sorted(glob.glob("/kaggle/input/**/b21_outputs/result.json", recursive=True))
assert found, "attach Run 21's output: Add Input -> Notebook Output -> your Run 21 notebook"
B21 = pathlib.Path(found[0]).parent
r21 = json.load(open(B21 / "result.json"))
assert r21["holdout_fingerprint"] == HOLDOUT_FP, "Run 21 used different holdout rows"
print("Run 21 output:", B21, "| holdout scores:", r21.get("holdout_scores"))

test = pd.read_csv(TEST_CSV)


def load_holdout(run_dir):
    p = run_dir / "holdout_probs.npy"
    return np.load(p) if p.exists() else None


def load_test(run_dir):
    p = run_dir / f"{TEST_CSV.stem}_probs.npy"     # llm_classifier: <input stem>_probs.npy
    if not p.exists():
        return None
    ids = pd.read_csv(run_dir / f"{TEST_CSV.stem}_ids.csv")["id"]
    assert ids.tolist() == test["id"].tolist(), f"{run_dir}: ids out of order"
    return np.load(p)


ho3 = {s: load_holdout(B21 / "runs" / f"{S}_ho_s{s}") for s in (42, 43)}
full3 = {s: load_test(B21 / "runs" / f"{S}_full_s{s}") for s in (42, 43)}
assert all(v is not None for v in ho3.values()), "Run 21's 3-epoch holdout probabilities missing"
print("Run 21 full fits found:", [s for s, v in full3.items() if v is not None])
download(MODEL)

## Stage 1: four epochs on the holdout

The same recipe as Run 21 with `--epochs 4`. The cosine schedule stretches to the longer
run, so this is not Run 21 plus one more epoch.

In [ ]:
EP4 = "--epochs 4"
jobs = [llm_job(MODEL, f"{S}_ep4_ho_s{s}", SPLIT / "train.csv", [SPLIT / "holdout.csv"], s,
                evalf=SPLIT / "holdout.csv", extra=EP4) for s in (42, 43)]
codes1 = run_queue(jobs, N_GPU, raise_on_fail=False, stop_at=HARD_STOP)
ho4 = {s: load_holdout(RUNS / f"{S}_ep4_ho_s{s}") for s in (42, 43)}
for s in (42, 43):
    m = RUNS / f"{S}_ep4_ho_s{s}" / "metrics.json"
    if m.exists():
        print(f"4-epoch seed {s} per-epoch holdout (diagnostic):",
              [round(h.get("eval_macro_f1", 0), 4) for h in json.load(open(m))["history"]])
print("exit codes:", codes1)

## Decision: 3 epochs, 4 epochs, or both

The candidate with the highest holdout macro-F1 wins. If the 4-epoch runs failed, `3ep`
wins by default. The paired bootstraps against `3ep` say whether the margin is more than
noise. Either way, the final model averages more seeds than Run 21's two.

In [ ]:
cands = {"3ep": np.mean(list(ho3.values()), 0)}
if all(v is not None for v in ho4.values()):
    cands["4ep"] = np.mean(list(ho4.values()), 0)
    cands["mix"] = np.mean(list(ho3.values()) + list(ho4.values()), 0)
for s, p in ho3.items():
    report(y_ho, p, f"3 epochs seed {s}")
for s, p in ho4.items():
    if p is not None:
        report(y_ho, p, f"4 epochs seed {s}")
scores = {k: report(y_ho, v, f"{k} averaged") for k, v in cands.items()}
boots = {}
for k in cands:
    if k != "3ep":
        b = paired_bootstrap(y_ho, cands[k], cands["3ep"])
        boots[f"{k} - 3ep"] = b
        print(f"{k} - 3ep  {b['diff']:+.4f}  CI [{b['ci95'][0]:+.4f}, {b['ci95'][1]:+.4f}]  "
              f"P(better) {b['p_better']:.2f}")
CHOICE = max(scores, key=scores.get)
print("\nchosen:", CHOICE)

## Stage 2: full fits on all 3,532 rows

The full fits for the chosen candidate. A 4-epoch full fit takes about 2 h on a T4, so
`4ep` queues a fourth seed only if the 11 h budget still allows it.

In [ ]:
def full(seed, ep4):
    tag = "ep4_" if ep4 else ""
    return llm_job(MODEL, f"{S}_{tag}full_s{seed}", SPLIT / "all.csv", [TEST_CSV], seed,
                   extra=EP4 if ep4 else "")


plan = {"3ep": [full(44, False), full(45, False)],
        "4ep": [full(42, True), full(43, True), full(44, True)],
        "mix": [full(42, True), full(43, True)]}[CHOICE]
hours_left = (HARD_STOP - time.time()) / 3600
if CHOICE == "4ep" and hours_left > 6.5:
    plan.append(full(45, True))
print(f"{hours_left:.1f} h left; queueing:", [j["name"] for j in plan])
codes2 = run_queue(plan, N_GPU, raise_on_fail=False, stop_at=HARD_STOP)
print("exit codes:", codes2, f"| {(time.time() - T_START) / 3600:.1f} h elapsed")

## Stage 3: the submission

The test probabilities of every finished model in the chosen candidate are averaged. Run 21's
3-epoch full fits are included for `3ep` and `mix`. The ZIP holds one bare `predictions.csv`
(`id,label`).

In [ ]:
import zipfile

members = {}
if CHOICE in ("3ep", "mix"):
    members.update({f"run21_3ep_s{s}": p for s, p in full3.items() if p is not None})
for j in plan:
    p = load_test(RUNS / j["name"])
    if p is not None:
        members[j["name"]] = p
assert members, "no full fit finished -- see the logs"
print("averaging", len(members), "models:", list(members))
probs = np.mean(list(members.values()), 0)
assert probs.shape == (len(test), 6), probs.shape

SUB = OUT / "submissions"
SUB.mkdir(exist_ok=True)
pred = pd.DataFrame({"id": test["id"], "label": [LABELS[i] for i in probs.argmax(1)]})
assert pred["id"].is_unique and set(pred["label"]) <= set(LABELS)
np.save(SUB / "b23_llm_test_probs.npy", probs)
pred.to_csv(SUB / "predictions_b23_llm.csv", index=False)
with zipfile.ZipFile(SUB / "RECOMMENDED_b23_llm.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(SUB / "predictions_b23_llm.csv", "predictions.csv")

prior = df["Hate Category"].value_counts(normalize=True)
counts = pred["label"].value_counts(normalize=True)
print("predicted/prior: " + "  ".join(f"{l[:5]} {counts.get(l, 0):.1%}/{prior[l]:.1%}" for l in LABELS))
if all(p is not None for p in full3.values()):
    run21 = np.mean(list(full3.values()), 0).argmax(1)
    print(f"agrees with Run 21's 2-seed LLM on {(run21 == probs.argmax(1)).mean():.1%} of test rows")

json.dump({"commit": head, "holdout_fingerprint": HOLDOUT_FP, "choice": CHOICE,
           "holdout_scores": scores, "bootstrap": boots, "members": list(members),
           "exit_codes": {**codes1, **codes2}},
          open(OUT / "result.json", "w"), indent=2)
print("wrote", SUB / "RECOMMENDED_b23_llm.zip")

## Record

In `docs/EXPERIMENTS.md`, record:
- the three candidates' holdout scores and the bootstrap lines
- the choice
- the models averaged
- the CodaBench score of `RECOMMENDED_b23_llm.zip`

The bar is `RECOMMENDED_b21_llm.zip` from Run 22, and the confirmed best before that is
`0.6410`.